# Phase 2D - BAI Judge Consistency (Colab)

Calibrate the five-key BAI GLM judge against the existing Fireworks GLM judge. This notebook reuses saved answers and makes no retrieval or generation calls.

Both P2 and P2-1S use the same frozen 20-question judge subset. BAI component results are stored separately; the Fireworks reference files are never modified.


In [ ]:
from pathlib import Path
REPO_URL='https://github.com/ThomasdeCarpio/Text-Mining---NewsQA-RAG.git'
REPO_COMMIT='aa3e4fcc852f67d0bdded470b2d8b1405b01bbb0'
SOURCE_ZIP=Path('/content/drive/MyDrive/newsqa_phase2/phase2d_p2_one_shot_results.zip')
RUN_ID='phase2d_bai_judge_consistency'
EXECUTE_BAI_JUDGE=False  # Set True after configuring all Colab secrets.
BAI_KEY_SECRET_NAMES=[f'BAI_API_KEY_{index}' for index in range(1,6)]
BAI_BASE_URL_SECRET_NAME='BAI_BASE_URL'
BAI_JUDGE_MODEL='glm-5.3-flash'  # Change only if BAI exposes a different exact model ID.
JUDGE_REASONING_EFFORT='low'
JUDGE_MAX_TOKENS=2048
SEED=42
METRICS=['answer_correctness','faithfulness','answer_relevancy','context_precision','context_recall']
ARMS={'p2_d5':'one_shot_screening__p2__d5','p2_1s_d5':'one_shot_screening__p2_1s__d5'}
MEAN_DELTA_LIMIT=0.03
MAE_LIMIT=0.10
SPEARMAN_MIN=0.80
DISCRETE_AGREEMENT_MIN=0.85


## 1. Environment and immutable source

Required Colab secrets: `BAI_API_KEY_1` through `BAI_API_KEY_5`, and `BAI_BASE_URL`. A GPU is optional but speeds up the local answer-relevancy embeddings.


In [ ]:
import hashlib, json, math, os, shutil, subprocess, sys, time, zipfile
from google.colab import drive, userdata
drive.mount('/content/drive')
RUNTIME_ROOT=Path('/content'); PROJECT_ROOT=RUNTIME_ROOT/'Text-Mining---NewsQA-RAG'; WORK_ROOT=RUNTIME_ROOT/RUN_ID
SOURCE_ROOT=WORK_ROOT/'source'; RUNS_ROOT=WORK_ROOT/'runs'; IDS_ROOT=WORK_ROOT/'question_ids'; RESULTS=WORK_ROOT/'results'; LOGS=WORK_ROOT/'logs'
DRIVE_OUTPUT=Path('/content/drive/MyDrive/newsqa_phase2'); DRIVE_OUTPUT.mkdir(parents=True,exist_ok=True)
def secret(name):
    try: return (userdata.get(name) or '').strip()
    except Exception: return ''
assert SOURCE_ZIP.exists(),f'Upload the screening ZIP to {SOURCE_ZIP}'
for path in [SOURCE_ROOT,RUNS_ROOT,IDS_ROOT,RESULTS,LOGS]: path.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(SOURCE_ZIP) as archive: archive.extractall(SOURCE_ROOT)
assert not REPO_COMMIT.startswith('SET_TO_'),'Commit and pin this notebook before execution'
if not PROJECT_ROOT.exists(): subprocess.run(['git','clone','--filter=blob:none',REPO_URL,str(PROJECT_ROOT)],check=True)
subprocess.run(['git','fetch','--depth=1','origin',REPO_COMMIT],cwd=PROJECT_ROOT,check=True,timeout=180)
subprocess.run(['git','checkout','--detach',REPO_COMMIT],cwd=PROJECT_ROOT,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt'],cwd=PROJECT_ROOT,check=True)
os.environ['PYTHONPATH']=os.pathsep.join([str(PROJECT_ROOT/'common'),str(PROJECT_ROOT/'app/backend'),os.environ.get('PYTHONPATH','')]).rstrip(os.pathsep)
os.environ.update({'HF_HOME':str(RUNTIME_ROOT/'hf_cache'),'TOKENIZERS_PARALLELISM':'false','OMP_NUM_THREADS':'1','MKL_NUM_THREADS':'1','PYTHONUNBUFFERED':'1','LANGCHAIN_TRACING_V2':'false','LANGSMITH_TRACING':'false'})
BAI_API_KEYS=[secret(name) for name in BAI_KEY_SECRET_NAMES]; BAI_BASE_URL=secret(BAI_BASE_URL_SECRET_NAME).rstrip('/')
if EXECUTE_BAI_JUDGE:
    assert all(BAI_API_KEYS),f'Configure all secrets: {BAI_KEY_SECRET_NAMES}'
    assert len(set(BAI_API_KEYS))==5,'Each BAI key slot must use a distinct account key'
    assert BAI_BASE_URL.startswith('https://'),f'Configure {BAI_BASE_URL_SECRET_NAME}'
print('Source SHA-256:',hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest())


In [ ]:
import pandas as pd, numpy as np
from IPython.display import display
def load_jsonl(path): return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]
def write_json(path,value): Path(path).write_text(json.dumps(value,indent=2,sort_keys=True)+'\n',encoding='utf-8')
def write_jsonl(path,rows): Path(path).write_text(''.join(json.dumps(row,sort_keys=True)+'\n' for row in rows),encoding='utf-8')
def latest_success(path):
    result={}
    for row in load_jsonl(path):
        if row.get('status')=='success': result[row['question_id']]=row
    return result
def run_command(command,label,env_overrides=None):
    command=[str(value) for value in command]; log_path=LOGS/f'{label}_{time.strftime("%Y%m%d_%H%M%S")}.log'
    print('$',' '.join(command),flush=True); print('Log:',log_path,flush=True)
    with log_path.open('w',encoding='utf-8') as log:
        env=os.environ.copy(); env.update(env_overrides or {})
        process=subprocess.Popen(command,cwd=PROJECT_ROOT,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
        for line in process.stdout: print(line,end='',flush=True); log.write(line); log.flush()
        code=process.wait()
    if code: raise subprocess.CalledProcessError(code,command)
    return log_path
def checkpoint():
    target=DRIVE_OUTPUT/f'{RUN_ID}_checkpoint.zip'; temporary=target.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED) as archive:
        for root_name in ['runs','question_ids','results','logs']:
            root=WORK_ROOT/root_name
            for path in root.rglob('*'):
                if path.is_file(): archive.write(path,path.relative_to(WORK_ROOT))
    temporary.replace(target); print('Checkpoint:',target,round(target.stat().st_size/2**20,1),'MiB'); return target


## 2. Validate and isolate the Fireworks reference runs


In [ ]:
judge_ids=json.loads((SOURCE_ROOT/'question_ids/judge_calibration.json').read_text()); assert len(judge_ids)==20 and len(set(judge_ids))==20
reference={}; calibration_runs={}
for arm,folder in ARMS.items():
    source_run=SOURCE_ROOT/'runs'/folder; assert source_run.exists(),source_run
    source_judges=latest_success(source_run/'judge_results.jsonl'); assert set(judge_ids)<=set(source_judges)
    assert all(set(row['scores'])==set(METRICS) for qid,row in source_judges.items() if qid in set(judge_ids))
    reference[arm]={qid:source_judges[qid] for qid in judge_ids}
    target=RUNS_ROOT/arm
    if not target.exists(): shutil.copytree(source_run,target)
    calibration_runs[arm]=target
write_json(IDS_ROOT/'judge_calibration.json',judge_ids)
display(pd.DataFrame([{'arm':arm,'questions':len(rows),'reference_provider':next(iter(rows.values()))['judge_provider'],'reference_model':next(iter(rows.values()))['judge_model']} for arm,rows in reference.items()]))


## 3. Run the five-key BAI judge

Keys 1–3 own metric groups. Keys 4–5 split questions for Context Precision, while every question retains all five ranked contexts. Partial JSONL files provide resumability.


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
def run_multi_key_arm(arm,run_dir):
    context_a=judge_ids[::2]; context_b=judge_ids[1::2]
    all_path=IDS_ROOT/f'{arm}_all.json'; a_path=IDS_ROOT/f'{arm}_context_a.json'; b_path=IDS_ROOT/f'{arm}_context_b.json'
    write_json(all_path,judge_ids); write_json(a_path,context_a); write_json(b_path,context_b)
    specs=[
        {'slot':1,'name':'faithfulness','metrics':['faithfulness'],'ids':all_path,'interval':30.0},
        {'slot':2,'name':'answer_correctness','metrics':['answer_correctness'],'ids':all_path,'interval':30.0},
        {'slot':3,'name':'relevancy_recall','metrics':['answer_relevancy','context_recall'],'ids':all_path,'interval':30.0},
        {'slot':4,'name':'context_precision_a','metrics':['context_precision'],'ids':a_path,'interval':65.0},
        {'slot':5,'name':'context_precision_b','metrics':['context_precision'],'ids':b_path,'interval':65.0},
    ]
    def worker(spec):
        result_file=f"judge_bai_key{spec['slot']}_{spec['name']}.jsonl"
        attempts_file=f"attempts_bai_key{spec['slot']}_{spec['name']}.jsonl"
        command=[sys.executable,'-u','scripts/judge_benchmark_predictions.py','--run-dir',run_dir,'--judge-provider','bai','--judge-model',BAI_JUDGE_MODEL,'--reasoning-effort',JUDGE_REASONING_EFFORT,'--judge-max-tokens',JUDGE_MAX_TOKENS,'--question-ids-file',spec['ids'],'--results-file',result_file,'--attempts-file',attempts_file,'--metrics',*spec['metrics'],'--batch-size',1,'--max-workers',1,'--min-batch-interval-seconds',spec['interval'],'--max-attempts',3,'--retry-failed','--require-complete-metrics','--progress']
        run_command(command,f"{arm}_key{spec['slot']}_{spec['name']}",{'BAI_API_KEY':BAI_API_KEYS[spec['slot']-1],'BAI_BASE_URL':BAI_BASE_URL})
        return spec,result_file
    completed=[]
    try:
        with ThreadPoolExecutor(max_workers=5) as pool:
            futures=[pool.submit(worker,spec) for spec in specs]
            for future in as_completed(futures): completed.append(future.result())
    except Exception:
        checkpoint(); raise
    partial={}
    for spec,result_file in completed:
        rows=latest_success(run_dir/result_file); expected=context_a if spec['slot']==4 else context_b if spec['slot']==5 else judge_ids
        missing=[qid for qid in expected if qid not in rows]; assert not missing,f"Incomplete {arm}/{spec['name']}: {missing[:5]}"
        for qid in expected: partial.setdefault(qid,[]).append((spec,rows[qid]))
    merged=[]
    for qid in judge_ids:
        components=partial[qid]; scores={}; usage={}; provenance=[]
        for spec,row in components:
            scores.update(row['scores']); provenance.append({'key_slot':spec['slot'],'worker':spec['name'],'metrics':spec['metrics'],'judge_fingerprint':row['judge_fingerprint']})
            for key,value in (row.get('batch_usage') or {}).items():
                if isinstance(value,(int,float)): usage[key]=usage.get(key,0)+value
        assert set(scores)==set(METRICS),(qid,scores)
        fingerprint=hashlib.sha256(json.dumps({'arm':arm,'question_id':qid,'provider':'bai-multi-key','model':BAI_JUDGE_MODEL,'components':provenance},sort_keys=True).encode()).hexdigest()
        merged.append({'question_id':qid,'status':'success','judge_fingerprint':fingerprint,'judge_provider':'bai-multi-key','judge_model':BAI_JUDGE_MODEL,'reasoning_effort':JUDGE_REASONING_EFFORT,'judge_max_tokens':JUDGE_MAX_TOKENS,'metrics':METRICS,'scores':scores,'missing_metrics':[],'batch_id':hashlib.sha256(f'{arm}:{qid}'.encode()).hexdigest()[:16],'batch_elapsed_ms':max(row.get('batch_elapsed_ms',0) for _,row in components),'batch_usage':usage,'attempt_count':max(row.get('attempt_count',1) for _,row in components),'component_provenance':provenance,'finished_at':max(row.get('finished_at','') for _,row in components)})
    output=run_dir/'judge_results_bai_multikey.jsonl'; write_jsonl(output,merged)
    write_json(run_dir/'bai_multi_key_manifest.json',{'schema_version':1,'arm':arm,'provider':'bai','model':BAI_JUDGE_MODEL,'reasoning_effort':JUDGE_REASONING_EFFORT,'key_slots':5,'rpm_limit_per_key':8,'allocation':[{'key_slot':s['slot'],'worker':s['name'],'metrics':s['metrics'],'minimum_batch_interval_seconds':s['interval']} for s in specs],'context_precision_partition':'alternating questions; all five contexts retained','secrets_recorded':False})
    return output
assert EXECUTE_BAI_JUDGE,'Set EXECUTE_BAI_JUDGE=True after validating the source and secrets'
candidate={}
for arm,run_dir in calibration_runs.items(): candidate[arm]=latest_success(run_multi_key_arm(arm,run_dir)); checkpoint()


## 4. Paired consistency analysis


In [ ]:
question_rows=[]
for arm in ARMS:
    predictions={row['question_id']:row for row in load_jsonl(calibration_runs[arm]/'predictions.jsonl')}
    assert set(judge_ids)==set(candidate[arm])
    for qid in judge_ids:
        article=predictions[qid]['article_key']
        for metric in METRICS:
            ref=float(reference[arm][qid]['scores'][metric]); cand=float(candidate[arm][qid]['scores'][metric])
            question_rows.append({'arm':arm,'question_id':qid,'article_key':article,'metric':metric,'fireworks':ref,'bai':cand,'delta_bai_minus_fireworks':cand-ref,'absolute_error':abs(cand-ref)})
question_frame=pd.DataFrame(question_rows); question_frame.to_csv(RESULTS/'judge_consistency_question_level.csv',index=False)
def cluster_bootstrap_ci(group,n_boot=5000):
    article_delta=group.groupby('article_key')['delta_bai_minus_fireworks'].mean().to_numpy(); rng=np.random.default_rng(SEED)
    boot=np.array([rng.choice(article_delta,size=len(article_delta),replace=True).mean() for _ in range(n_boot)])
    return np.quantile(boot,[0.025,0.975])
summary=[]
for (arm,metric),group in question_frame.groupby(['arm','metric']):
    lo,hi=cluster_bootstrap_ci(group); rho=group['fireworks'].corr(group['bai'],method='spearman')
    discrete=metric in {'faithfulness','context_precision','context_recall'}
    within=(group['absolute_error']<=0.10+1e-12).mean()
    gate=abs(group['delta_bai_minus_fireworks'].mean())<=MEAN_DELTA_LIMIT and group['absolute_error'].mean()<=MAE_LIMIT and (not math.isnan(rho) and rho>=SPEARMAN_MIN) and (not discrete or within>=DISCRETE_AGREEMENT_MIN)
    summary.append({'arm':arm,'metric':metric,'n':len(group),'articles':group['article_key'].nunique(),'fireworks_mean':group['fireworks'].mean(),'bai_mean':group['bai'].mean(),'mean_delta':group['delta_bai_minus_fireworks'].mean(),'mae':group['absolute_error'].mean(),'spearman':rho,'agreement_within_0.10':within,'ci95_low':lo,'ci95_high':hi,'screening_gate_pass':gate})
summary_frame=pd.DataFrame(summary); summary_frame.to_csv(RESULTS/'judge_consistency_metric_summary.csv',index=False)
disagreements=question_frame[question_frame['absolute_error']>0.10].sort_values(['absolute_error'],ascending=False); disagreements.to_csv(RESULTS/'judge_consistency_disagreements.csv',index=False)
coverage=sum(len(candidate[arm]) for arm in ARMS)/(len(ARMS)*len(judge_ids))
write_json(RESULTS/'judge_consistency_decision.json',{'schema_version':1,'coverage':coverage,'questions_per_arm':len(judge_ids),'arms':list(ARMS),'answer_instances':len(ARMS)*len(judge_ids),'all_metric_gates_pass':bool(summary_frame['screening_gate_pass'].all()),'thresholds':{'absolute_mean_delta_max':MEAN_DELTA_LIMIT,'mae_max':MAE_LIMIT,'spearman_min':SPEARMAN_MIN,'discrete_agreement_within_0.10_min':DISCRETE_AGREEMENT_MIN},'interpretation':'Operational calibration screen only; n=20 per arm does not establish statistical equivalence.'})
display(summary_frame.sort_values(['metric','arm'])); print('Coverage:',coverage,'| disagreements > 0.10:',len(disagreements)); display(disagreements.head(30))


## 5. Export


In [ ]:
manifest={'schema_version':1,'repo_commit':REPO_COMMIT,'source_zip':SOURCE_ZIP.name,'source_sha256':hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest(),'reference_provider':'fireworks','candidate_provider':'bai-multi-key','candidate_model':BAI_JUDGE_MODEL,'reasoning_effort':JUDGE_REASONING_EFFORT,'question_ids_sha256':hashlib.sha256(json.dumps(judge_ids,sort_keys=True,separators=(',',':')).encode()).hexdigest(),'arms':ARMS,'metrics':METRICS,'generated_at':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())}
write_json(RESULTS/'calibration_manifest.json',manifest)
bundle=DRIVE_OUTPUT/f'{RUN_ID}_results.zip'; temporary=bundle.with_suffix('.zip.tmp')
with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for root_name in ['runs','question_ids','results','logs']:
        root=WORK_ROOT/root_name
        for path in root.rglob('*'):
            if path.is_file(): archive.write(path,path.relative_to(WORK_ROOT))
temporary.replace(bundle); checkpoint(); print('Results:',bundle,round(bundle.stat().st_size/2**20,1),'MiB')
